# 9.5장 콘텐츠 기반 필터링 실습 - TMDB 5000 영화 데이터 세트

## 데이터 로딩 및 가공

In [1]:
!pip install --upgrade kagglehub[pandas-datasets]

In [2]:
import pandas as pd
import numpy as np
import warnings; warnings.filterwarnings('ignore')

movies = pd.read_csv('/content/drive/MyDrive/Sample_data/tmdb_movies/tmdb_5000_movies.csv')
print(movies.shape)
movies.head(1)

(4803, 20)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [3]:
movies_df = movies[['id','title','genres','vote_average','vote_count','popularity','keywords','overview']]

- genres, keywords 과 같은 칼럼들은 파이썬 리스트 내부에 여러개의 딕셔너리가 있는 형태의 문자열임 ; 한번에 여러개의 값 표기
- genres
: 여러개의 개별 장르 데이터 가짐, 개별장르의 명칭은 'name'이라는 키로 추출 가능
- literal_eval() 함수 : list[dict1,dict2] 객체로 만들 수 있음.

In [4]:
pd.set_option('max_colwidth',100)
movies_df[['genres','keywords']][:1]

,genres,keywords
0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""name"": ""future""}, {""id"": 3386, ""name"": ""sp..."


In [5]:
from ast import literal_eval
# genres : 문자열 X, 실제 리스트 내부에 여러 장르 딕셔너리로 구성된 객체
movies_df['genres']=movies_df['genres'].apply(literal_eval)
movies_df['keywords']=movies_df['keywords'].apply(literal_eval)

In [6]:
movies_df['genres']=movies_df['genres'].apply(lambda x :[y['name'] for y in x])
movies_df['keywords']=movies_df['keywords'].apply(lambda x :[y['name'] for y in x])
movies_df[['genres','keywords']][:1]

,genres,keywords
0,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colony, society, space travel, futuristic, romance, spa..."


## 장르 콘텐츠 유사도 측정

1. genres 문자열 변경 > CountVectorizer로 피처 벡터화
2. genres 문자열을 피처벡터화 행렬로 변환한 데이터 세트를 코사인 유사도를 통해 비교 / 데이터 세트의 레코드별로 타 레코드와 장르에서 코사인 유사도값을 가지는 객체 생성
3. 장르 유사도가 높은 영화 중 평점이 높은 순으로 영화 추천

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

# CountVectorizer 적용 위해 공백문자로 word 단위가 구분되는 문자열로 변환
movies_df['genres_literal'] = movies_df['genres'].apply(lambda x : (' ').join(x))
count_vect = CountVectorizer(min_df=0.00000000001, ngram_range=(1,2))
genre_mat = count_vect.fit_transform(movies_df['genres_literal'])
print(genre_mat.shape)

(4803, 276)


- movies_df의 행별 장르 우사도 값 지니는 행렬 생성

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

genre_sim = cosine_similarity(genre_mat, genre_mat)
print(genre_sim.shape)
print(genre_sim[:2])

(4803, 4803)
[[1.         0.59628479 0.4472136  ... 0.         0.         0.        ]
 [0.59628479 1.         0.4        ... 0.         0.         0.        ]]


- genre_sim 객체의 기준 행별로 비교 대상이 되는 행의 유사도 값이 높은 순으로 정렬된 행렬의 위치 인덱스 값 추출
- ⬇0번 레코드의 경우 자신인 0번 레코드를 제외하면 46,3494 순으로 유사도가 높고 가장 유사도가 낮은 인덱스는 2031

In [9]:
genre_sim_sorted_ind = genre_sim.argsort()[:, ::-1]
print(genre_sim_sorted_ind[:1])

[[   0   46 3494 ... 3331 3333 2031]]


## 장르 콘텐츠 필터링을 이용한 영화 추천

### find_sim_movie()
- input : movies_df,genre_sorted_ind, 고객의 추천 기준이 되는 영화제목, 추천할 영화 건수
- return : 추천영화정보를 가지는 df


In [10]:
def find_sim_movie(df,sorted_ind,title_name,top_n=10):
  # movies_df에서 'title'칼럼이 입력된 title_name값인 df 추출
  title_movie = df[df['title']==title_name]

  # title_name가진 df의 인덱스객체 > ndarray
  # genre_sim_sorted_ind 객체에서 유사도순으로 top_n개의 index 추출
  title_index = title_movie.index.values
  similar_indexes = sorted_ind[title_index,:(top_n)]

  # top_n 인텍스 출력(2차원데이터) > 1차원 ndarray로 변환
  print(similar_indexes)
  similar_indexes = similar_indexes.reshape(-1)

  return df.iloc[similar_indexes]

In [11]:
similar_movies = find_sim_movie(movies_df, genre_sim_sorted_ind, 'The Godfather',10)
similar_movies[['title', 'vote_average']]

[[1881 3378 3866 1370 1464  588 3887 3594 2839  892]]


,title,vote_average
1881,The Shawshank Redemption,8.5
3378,Auto Focus,6.1
3866,City of God,8.1
1370,21,6.5
1464,Black Water Transit,0.0
588,Wall Street: Money Never Sleeps,5.8
3887,Trainspotting,7.8
3594,Spring Breakers,5.0
2839,Rounders,6.9
892,Casino,7.8


- 고객에게 왜추천하는지 이해하기 어려운 영화 有 > 평점 0.0
- 많은 후보군 선정 > 평점에 따라 필터링 후 최종 추천

- vote_average : 0~10, 특정고객이 만점을 부여해 왜곡된 데이터를 가지고 있을 수 있음. > vote_count 고려한 가중 평점 생성
- 가중평점 = (vote_count/(vote_count+최소투표횟수)) * 개별영화평균평점 + (vote_count/(vote_count+최소투표횟수)) * 전체 영화 평균 평점
- 가중치 : (vote_count/(vote_count+최소투표횟수))


In [12]:
movies_df[['title','vote_average','vote_count']].sort_values('vote_average', ascending=False)[:10]

,title,vote_average,vote_count
4662,Little Big Top,10.0,1
3519,Stiff Upper Lips,10.0,1
4045,"Dancer, Texas Pop. 81",10.0,1
4247,Me You and Five Bucks,10.0,2
3992,Sardaarji,9.5,2
2386,One Man's Hero,9.3,2
1881,The Shawshank Redemption,8.5,8205
2970,There Goes My Baby,8.5,2
3337,The Godfather,8.4,5893
2796,The Prisoner of Zenda,8.4,11


In [13]:
C = movies_df['vote_average'].mean()
m = movies_df['vote_count'].quantile(0.6) #상위60퍼
print('C:',round(C,3), 'm:',round(m,3))

C: 6.092 m: 370.2


In [14]:
percentile = 0.6
m = movies_df['vote_count'].quantile(percentile)
C = movies_df['vote_average'].mean()

def weighted_vote_average(record):
    v = record['vote_count']
    R = record['vote_average']

    return ( (v/(v+m)) * R ) + ( (m/(m+v)) * C )

movies_df['weighted_vote'] = movies_df.apply(weighted_vote_average, axis=1)

In [15]:
movies_df[['title','vote_average','weighted_vote','vote_count']].sort_values('weighted_vote',
                                                                          ascending=False)[:10]

,title,vote_average,weighted_vote,vote_count
1881,The Shawshank Redemption,8.5,8.396052,8205
3337,The Godfather,8.4,8.263591,5893
662,Fight Club,8.3,8.216455,9413
3232,Pulp Fiction,8.3,8.207102,8428
65,The Dark Knight,8.2,8.136930,12002
1818,Schindler's List,8.3,8.126069,4329
3865,Whiplash,8.3,8.123248,4254
809,Forrest Gump,8.2,8.105954,7927
2294,Spirited Away,8.3,8.105867,3840
2731,The Godfather: Part II,8.3,8.079586,3338


In [16]:
# 콘텐츠 기반 필터링
def find_sim_movie2(df, sorted_ind, title_name, top_n=10):
    title_movie = df[df['title'] == title_name]
    title_index = title_movie.index.values

    # top_n의 2배 정도 유사성이 높은 index 추출
    similar_indexes = sorted_ind[title_index, :(top_n*2)]
    similar_indexes = similar_indexes.reshape(-1)
    # 기준 영화 index  제외
    similar_indexes = similar_indexes[similar_indexes != title_index]

    # top_n의 2배에 해당하는 후보군에서 weighted_vote 높은 순으로 top_n 만큼 추출
    return df.iloc[similar_indexes].sort_values('weighted_vote', ascending=False)[:top_n]

similar_movies = find_sim_movie2(movies_df, genre_sim_sorted_ind, 'The Godfather',10)
similar_movies[['title', 'vote_average', 'weighted_vote']]

,title,vote_average,weighted_vote
1881,The Shawshank Redemption,8.5,8.396052
2731,The Godfather: Part II,8.3,8.079586
1847,GoodFellas,8.2,7.976937
3866,City of God,8.1,7.759693
1663,Once Upon a Time in America,8.2,7.657811
3887,Trainspotting,7.8,7.591009
883,Catch Me If You Can,7.7,7.557097
892,Casino,7.8,7.423040
4041,This Is England,7.4,6.739664
1149,American Hustle,6.8,6.717525


# 9.6장 아이템 기반 최근접 이웃 협업 필터링 실습

##데이터 가공 및 변환

- 협업 필터링 : 사용자와 아이템 간의 평점에 기반해 추천하는 시스템
- 행:사용자, 칼럼:영화, 값: 평점인 데이터 세트로 변경

In [17]:
import pandas as pd
import numpy as np

movies = pd.read_csv('/content/drive/MyDrive/Sample_data/ml-latest-small/ml_latest_small_movies.csv')
ratings = pd.read_csv('/content/drive/MyDrive/Sample_data/ml-latest-small/ml_latest_small_ratings.csv')
print(movies.shape)
print(ratings.shape)

(9742, 3)
(100835, 4)


In [18]:
ratings.columns = ['userId','movieId','rating','timestamp']

- Nan값 : 사용자가 평점을 매기지 않은 영화가 컬럼으로 변환되며 값 할당
- Nan > 0

In [19]:
ratings=ratings[['userId','movieId','rating']]
ratings_matrix = ratings.pivot_table('rating',index='userId',columns='movieId')
ratings_matrix.head(3)

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
# title > movies와 조인
rating_movies = pd.merge(ratings,movies,on='movieId')
# columns='title'칼럼으로 피벗 수행
ratings_matrix = rating_movies.pivot_table('rating',index='userId',columns='title')

In [21]:
# Nan > 0
ratings_matrix = ratings_matrix.fillna(0)
ratings_matrix.head()

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 영화간 유사도 산출

- 영화간의 유사도 : 코사인유사도(cosine_similarity())
- ratings_matrix에 적용 시 영화간 ❌, 사용자간 유사도 생성 > transpose로 데이터 행과 열 위치 변경

In [22]:
ratings_matrix_T = ratings_matrix.transpose()
ratings_matrix_T.head(3)

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
'Hellboy': The Seeds of Creation (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Round Midnight (1986),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
from sklearn.metrics.pairwise import cosine_similarity

item_sim = cosine_similarity(ratings_matrix_T, ratings_matrix_T)

# cosine_similarity() 로 반환된 넘파이 행렬을 영화명을 매핑하여 DataFrame으로 변환
item_sim_df = pd.DataFrame(data=item_sim, index=ratings_matrix.columns,
                          columns=ratings_matrix.columns)
print(item_sim_df.shape)
item_sim_df.head(3)

(9719, 9719)


title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),1.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.141653,0.0,...,0.0,0.342055,0.543305,0.707107,0.0,0.0,0.139431,0.327327,0.0,0.0
'Hellboy': The Seeds of Creation (2004),0.0,1.000000,0.707107,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Round Midnight (1986),0.0,0.707107,1.000000,0.0,0.0,0.0,0.176777,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0


In [24]:
item_sim_df["Godfather, The (1972)"].sort_values(ascending=False)[:6]

,"Godfather, The (1972)"
title,
"Godfather, The (1972)",1.000000
"Godfather: Part II, The (1974)",0.821773
Goodfellas (1990),0.664841
One Flew Over the Cuckoo's Nest (1975),0.620536
Star Wars: Episode IV - A New Hope (1977),0.595317
Fargo (1996),0.588614


In [25]:
item_sim_df["Inception (2010)"].sort_values(ascending=False)[1:6]

,Inception (2010)
title,
"Dark Knight, The (2008)",0.727263
Inglourious Basterds (2009),0.646103
Shutter Island (2010),0.617736
"Dark Knight Rises, The (2012)",0.617504
Fight Club (1999),0.615417


## 아이템 기반 인접 이웃 협업 필터링으로 **개인화**된 영화 추천

$$\hat{R}_{u,i} = \sum(S_{(i,N)}*R_{(u,N)}) / \sum(|S_{(i,N)}|) $$

- $\hat{R}$ : 개인화된 예측 평점값
- S : 아이템 i와 가장 유사도가 높은 top-n개 아이템의 유사도 벡터
- R : 사용자 u 아이템 i와 가장 유사도가 높은 Top-N개 아이템에 대한 실제 평점 벡터

- 예측 평점이 실제 평점과 영화의 코사인 유사도를 내적(+백터합으로 나눔)한 것이기 때매 실제 평점보다 작을 수 있음
- 예측 평가지표 MSE : get_mse() 사용자 정의 함수

In [26]:
def predict_rating(ratings_arr, item_sim_arr ):
    ratings_pred = ratings_arr.dot(item_sim_arr)/ np.array([np.abs(item_sim_arr).sum(axis=1)])
    return ratings_pred

In [27]:
ratings_pred = predict_rating(ratings_matrix.values , item_sim_df.values)
ratings_pred_matrix = pd.DataFrame(data=ratings_pred, index= ratings_matrix.index,
                                   columns = ratings_matrix.columns)
ratings_pred_matrix.head(3)

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.069675,0.571866,0.319053,0.227055,0.206958,0.192404,0.248972,0.102214,0.156064,0.177548,...,0.113120,0.180250,0.133235,0.127794,0.006179,0.211527,0.191956,0.134909,0.291970,0.720347
2,0.018260,0.042744,0.018861,0.000000,0.000000,0.035995,0.013413,0.002314,0.032213,0.014863,...,0.015640,0.020855,0.020119,0.015745,0.049983,0.014876,0.021616,0.024528,0.017563,0.000000
3,0.011884,0.030279,0.064437,0.003762,0.003749,0.002721,0.014625,0.002085,0.005666,0.006272,...,0.006923,0.011665,0.011800,0.012225,0.000000,0.008194,0.007017,0.009229,0.010420,0.084501


In [28]:
from sklearn.metrics import mean_squared_error

# 사용자가 평점을 부여한 영화에 대해서만 예측 성능 평가 MSE 를 구함.
def get_mse(pred, actual):
    # Ignore nonzero terms.
    pred = pred[actual.nonzero()].flatten()
    actual = actual[actual.nonzero()].flatten()
    return mean_squared_error(pred, actual)

print('아이템 기반 모든 인접 이웃 MSE: ', get_mse(ratings_pred, ratings_matrix.values ))

아이템 기반 모든 인접 이웃 MSE:  9.895359667092327


In [29]:
def predict_rating_topsim(ratings_arr, item_sim_arr, n=20):
    # 사용자-아이템 평점 행렬 크기와 같은 예측 행렬 0으로 초기화
    pred = np.zeros(ratings_arr.shape)

    # 사용자-아이템 평점 행렬의 열 크기만큼 반복 : 데이터크면 매우 오래걸림
    for col in range(ratings_arr.shape[1]):
        # 유사도 행렬 > 유사도가 큰 순 n개 데이터 행렬의 index 반환
        top_n_items = [np.argsort(item_sim_arr[:, col])[:-n-1:-1]]
        # 예측 평점 계산
        for row in range(ratings_arr.shape[0]):
            pred[row, col] = item_sim_arr[col, :][top_n_items].dot(ratings_arr[row, :][top_n_items].T)
            pred[row, col] /= np.sum(np.abs(item_sim_arr[col, :][top_n_items]))
    return pred

In [30]:
ratings_pred = predict_rating_topsim(ratings_matrix.values , item_sim_df.values, n=20)
print('아이템 기반 인접 TOP-20 이웃 MSE: ', get_mse(ratings_pred, ratings_matrix.values ))


# 행렬 > DataFrame
ratings_pred_matrix = pd.DataFrame(data=ratings_pred, index= ratings_matrix.index,
                                   columns = ratings_matrix.columns);

아이템 기반 인접 TOP-20 이웃 MSE:  3.6945760072593754


In [31]:
user_rating_id = ratings_matrix.loc[9, :]
user_rating_id[ user_rating_id > 0].sort_values(ascending=False)[:10]

,9
title,
Adaptation (2002),5.0
Austin Powers in Goldmember (2002),5.0
Back to the Future (1985),5.0
Citizen Kane (1941),5.0
"Lord of the Rings: The Fellowship of the Ring, The (2001)",5.0
"Lord of the Rings: The Two Towers, The (2002)",5.0
"Producers, The (1968)",5.0
Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981),5.0
Elling (2001),4.0


In [32]:
def get_unseen_movies(ratings_matrix, userId):
    # userId > Series
    # 반환된 user_rating 은 영화명(title)을 index로 가지는 Series 객체임.
    user_rating = ratings_matrix.loc[userId,:]

    # user_rating > 0 : 기존에 관람한 영화. 대상 index를 추출 > list 객체
    already_seen = user_rating[ user_rating > 0].index.tolist()

    # 모든 영화명을 list 객체로 만듬.
    movies_list = ratings_matrix.columns.tolist()

    # already_seen에 해당하는 movie는 movies_list에서 제거
    unseen_list = [ movie for movie in movies_list if movie not in already_seen]

    return unseen_list

In [42]:
def recomm_movie_by_userid(pred_df, userId, unseen_list, top_n=10):
    # 예측 평점 DF에서 사용자id index와 unseen_list로 들어온 영화명 컬럼추출
    # > 높은 순 정렬.
    recomm_movies = pred_df.loc[userId, unseen_list].sort_values(ascending=False)[:top_n]
    return recomm_movies

# 사용자 관람X 영화명 추출
unseen_list = get_unseen_movies(ratings_matrix, 9)

# 아이템 기반 인접 이웃 협업 필터링 : 영화 추천
recomm_movies = recomm_movie_by_userid(ratings_pred_matrix, 9, unseen_list, top_n=10)

# 행렬 > DF
recomm_movies = pd.DataFrame(data=recomm_movies.values,index=recomm_movies.index,columns=['pred_score'])
recomm_movies

,pred_score
title,
Rear Window (1954),5.704191
"South Park: Bigger, Longer and Uncut (1999)",5.454730
Rounders (1998),5.296919
Blade Runner (1982),5.244503
Roger & Me (1989),5.190057
Gattaca (1997),5.184963
Ben-Hur (1959),5.130873
Rosencrantz and Guildenstern Are Dead (1990),5.088743
"Big Lebowski, The (1998)",5.039282


# 9.7장 행렬 분해를 이용한 잠재 요인 협업 필터링 실습

Null data 多> SGD/ALS (SVD,NMF ❌)

### matrix_factorization 함수 ( 행렬 분해 함수 )
- input : R,K,steps,learning_rate,r_lambda
- R: 실제 행렬
- K: 잠재요인수
- learning_rate: 학습률
- r_lambda : L2 규제 계수

In [34]:
import numpy as np
from sklearn.metrics import mean_squared_error

def get_rmse(R, P, Q, non_zeros):
    error = 0
    # P * Q.T > 예측 R
    full_pred_matrix = np.dot(P, Q.T)

    # 실제 R 행렬에서 널이 아닌 값의 위치 인덱스 추출
    # > 실제 R - 예측 RMSE 추출
    x_non_zero_ind = [non_zero[0] for non_zero in non_zeros]
    y_non_zero_ind = [non_zero[1] for non_zero in non_zeros]
    R_non_zeros = R[x_non_zero_ind, y_non_zero_ind]

    full_pred_matrix_non_zeros = full_pred_matrix[x_non_zero_ind, y_non_zero_ind]

    mse = mean_squared_error(R_non_zeros, full_pred_matrix_non_zeros)
    rmse = np.sqrt(mse)

    return rmse

In [35]:
def matrix_factorization(R, K, steps=200, learning_rate=0.01, r_lambda = 0.01):
    num_users, num_items = R.shape
    # P Q 크기 지정/ 정규분포 랜덤값입력
    np.random.seed(1)
    P = np.random.normal(scale=1./K, size=(num_users, K))
    Q = np.random.normal(scale=1./K, size=(num_items, K))

    # R > 0 인 행 위치, 열 위치, 값을 non_zeros 리스트 객체에 저장.
    non_zeros = [ (i, j, R[i,j]) for i in range(num_users) for j in range(num_items) if R[i,j] > 0 ]

    # SGD기법으로 P, Q 계속 업데이트.
    for step in range(steps):
        for i, j, r in non_zeros:
            # 오류 : 실제 값 - 예측 값
            eij = r - np.dot(P[i, :], Q[j, :].T)
            # Regularization을 반영한 SGD 업데이트 공식 적용
            P[i,:] = P[i,:] + learning_rate*(eij * Q[j, :] - r_lambda*P[i,:])
            Q[j,:] = Q[j,:] + learning_rate*(eij * P[i, :] - r_lambda*Q[j,:])

        rmse = get_rmse(R, P, Q, non_zeros)
        if (step % 10) == 0 :
            print("### iteration step : ", step," rmse : ", rmse)

    return P, Q

In [36]:
import pandas as pd
import numpy as np
movies = pd.read_csv('/content/drive/MyDrive/Sample_data/ml-latest-small/ml_latest_small_movies.csv')
ratings = pd.read_csv('/content/drive/MyDrive/Sample_data/ml-latest-small/ml_latest_small_ratings.csv')
ratings.columns = ['userId', 'movieId', 'rating','timestamp']
ratings = ratings[['userId', 'movieId', 'rating']]
ratings_matrix = ratings.pivot_table('rating', index='userId', columns='movieId')

# title 컬럼을 얻기 이해 movies 와 조인 수행
rating_movies = pd.merge(ratings, movies, on='movieId')

# columns='title' 로 title 컬럼으로 pivot 수행.
ratings_matrix = rating_movies.pivot_table('rating', index='userId', columns='title')

In [37]:
P, Q = matrix_factorization(ratings_matrix.values, K=50, steps=200, learning_rate=0.01, r_lambda = 0.01)
pred_matrix = np.dot(P, Q.T)

### iteration step :  0  rmse :  2.902361483117536
### iteration step :  10  rmse :  0.7335761698317158
### iteration step :  20  rmse :  0.5115603383967597
### iteration step :  30  rmse :  0.3726213866510964
### iteration step :  40  rmse :  0.2960862274283514
### iteration step :  50  rmse :  0.2520402550097047
### iteration step :  60  rmse :  0.22488030600834652
### iteration step :  70  rmse :  0.20685967417903406
### iteration step :  80  rmse :  0.19413888540756688
### iteration step :  90  rmse :  0.18470501296551115
### iteration step :  100  rmse :  0.17743297964022212
### iteration step :  110  rmse :  0.17165553771539724
### iteration step :  120  rmse :  0.16695470884206962
### iteration step :  130  rmse :  0.16305548283230267
### iteration step :  140  rmse :  0.1597691912320195
### iteration step :  150  rmse :  0.15696188229537253
### iteration step :  160  rmse :  0.1545357555855267
### iteration step :  170  rmse :  0.1524177353314282
### iteration step :  180  rmse

In [38]:
# 행렬 > DF
ratings_pred_matrix = pd.DataFrame(data=pred_matrix, index= ratings_matrix.index,
                                   columns = ratings_matrix.columns)

ratings_pred_matrix.head(3)

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,3.053710,4.093457,3.565621,4.501866,3.979732,1.270481,3.610400,2.332452,5.079653,3.972328,...,1.404236,4.216634,3.707188,2.721879,2.787248,3.472398,3.245085,2.159882,4.009836,0.859327
2,3.170378,3.657983,3.308636,4.166149,4.311504,1.275675,4.238628,1.900196,3.393151,3.647428,...,0.973841,3.528000,3.361705,2.672623,2.404345,4.231644,2.911773,1.634416,4.134683,0.725547
3,2.306473,1.658783,1.443389,2.208475,2.228921,0.780457,1.995356,0.924486,2.975363,2.551171,...,0.520532,1.709802,2.281183,1.782552,1.635064,1.318541,2.887611,1.042975,2.294125,0.396788


In [41]:
unseen_list = get_unseen_movies(ratings_matrix, 9)

# 아이템 기반의 인접 이웃 협업 필터링으로 영화 추천
recomm_movies = recomm_movie_by_userid(ratings_pred_matrix, 9, unseen_list, top_n=10)

# 평점 데이타를 DataFrame으로 생성.
recomm_movies = pd.DataFrame(data=recomm_movies.values,index=recomm_movies.index,columns=['pred_score'])
recomm_movies

,pred_score
title,
Rear Window (1954),5.704191
"South Park: Bigger, Longer and Uncut (1999)",5.454730
Rounders (1998),5.296919
Blade Runner (1982),5.244503
Roger & Me (1989),5.190057
Gattaca (1997),5.184963
Ben-Hur (1959),5.130873
Rosencrantz and Guildenstern Are Dead (1990),5.088743
"Big Lebowski, The (1998)",5.039282
